<img src="./img/uva_seal.png">  

## Stateful Streaming with transformWithState
### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: November 15, 2026
---

### SOURCES

https://spark.apache.org/docs/latest/streaming/structured-streaming-transform-with-state.html

https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.GroupedData.transformWithStateInPandas.html

Data: daily AMZN and MSFT prices, Feb 2022 to Feb 2023 (`03_dataframes_and_spark_sql/content_sql_and_dataframes/amzn_msft_prices.csv`)

### ABOUT

Built-in streaming aggregations (`groupBy().count()`, windows) cover a lot, but some logic needs **custom memory per key** that lives across micro-batches. For example: *"alert me each time a stock falls another 10% below its running peak."* That needs, per ticker, the peak so far, the previous close, and a few recent prices.

**transformWithState** (Spark 4.0+) is Spark's general-purpose API for this. You write a `StatefulProcessor` class, and Spark calls it once per key per micro-batch with that key's new rows. The processor keeps named, typed **state variables** that Spark stores in the checkpoint (in RocksDB), so they survive restarts.

| State variable | Holds | Used here for |
|---|---|---|
| `ValueState` | one value | running peak, peak date, previous close |
| `ListState` | a list | the last 5 closes (moving average) |
| `MapState` | a key/value map | (not used) |

It also supports timers and TTL (state expiry), which we touch on at the end. In PySpark the API comes in two flavors: `transformWithStateInPandas` (rows arrive as pandas DataFrames, used here) and the row-based `transformWithState`. The processor class looks the same in both.

### OBJECTIVES
- Replay a static dataset as a stream, one file per micro-batch
- Write a `StatefulProcessor` with `ValueState` and `ListState`
- Apply it per key with `groupBy(...).transformWithStateInPandas(...)`
- Stop and restart a query and see its state survive in the checkpoint
---

### SETUP

transformWithState requires the **RocksDB state store**, so we set it when building the session. It also needs `pandas` and `pyarrow`.

In [ ]:
import os
import shutil
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.streaming import StatefulProcessor, StatefulProcessorHandle

spark = (SparkSession.builder
    .appName("TransformWithState-Demo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.streaming.stateStore.providerClass",
            "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

PRICES_CSV = "../../03_dataframes_and_spark_sql/content_sql_and_dataframes/amzn_msft_prices.csv"
WORK_DIR = os.path.abspath("tws_workspace")
LANDING = os.path.join(WORK_DIR, "landing")        # the stream's input folder
OUTPUT = os.path.join(WORK_DIR, "signals")         # parquet sink
CHECKPOINT = os.path.join(WORK_DIR, "checkpoint")  # offsets + RocksDB state

shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(LANDING)

print(f"Spark {spark.version}, workspace at {WORK_DIR}")

---

### 1. Turning a CSV into a Stream

We split the year of prices into **one file per month**. The streaming file source with `maxFilesPerTrigger=1` then treats each month as one micro-batch, as if the data were arriving over time. `land_months()` drops month files into the landing folder (oldest first).

In [ ]:
prices_pdf = pd.read_csv(PRICES_CSV)
prices_pdf["month"] = prices_pdf["date"].str[:7]
MONTHS = sorted(prices_pdf["month"].unique())
print(f"{len(prices_pdf)} rows, {len(MONTHS)} months: {MONTHS[0]} .. {MONTHS[-1]}")

def land_months(months):
    """Write one CSV per month into the landing folder, oldest first."""
    for m in months:
        path = os.path.join(LANDING, f"prices_{m}.csv")
        prices_pdf[prices_pdf["month"] == m].drop(columns="month").to_csv(path, index=False)
        t = pd.Timestamp(m + "-01").timestamp()   # file source orders files by modification time
        os.utime(path, (t, t))
    print("landing folder:", sorted(os.listdir(LANDING)))

prices_pdf.head()

---

### 2. The StatefulProcessor

For each ticker we track:

- **peak** (`ValueState`): highest close so far, its date, and the previous close
- **recent** (`ListState`): the last 5 closes, for a 5-day moving average

and emit one output row per input row, with a `signal` when the stock makes a **new high** or crosses another **10% drawdown** level below its peak.

Spark calls three methods:

| Method | Called | We use it to |
|---|---|---|
| `init(handle)` | once per partition when the query starts | declare the state variables |
| `handleInputRows(key, rows, timerValues)` | once per key per micro-batch that has rows for it | read state, process rows, update state, yield output |
| `close()` | at shutdown | (nothing) |

In [ ]:
WINDOW = 5

PEAK_SCHEMA = StructType([
    StructField("peak_close", DoubleType()),
    StructField("peak_date", StringType()),
    StructField("last_close", DoubleType()),
])
RECENT_SCHEMA = StructType([StructField("close", DoubleType())])

OUTPUT_SCHEMA = StructType([
    StructField("ticker", StringType()),
    StructField("date", StringType()),
    StructField("close", DoubleType()),
    StructField("sma_5", DoubleType()),
    StructField("peak_close", DoubleType()),
    StructField("peak_date", StringType()),
    StructField("drawdown_pct", DoubleType()),
    StructField("signal", StringType()),
])


class DrawdownTracker(StatefulProcessor):

    def init(self, handle: StatefulProcessorHandle) -> None:
        self.peak = handle.getValueState("peak", PEAK_SCHEMA)
        self.recent = handle.getListState("recent", RECENT_SCHEMA)

    def handleInputRows(self, key, rows, timerValues):
        (ticker,) = key
        # this batch's rows for this ticker; order within a batch is not guaranteed
        batch = pd.concat(list(rows)).sort_values("date")

        # 1. load state (empty the first time we see this ticker)
        if self.peak.exists():
            peak_close, peak_date, last_close = self.peak.get()
        else:
            peak_close, peak_date, last_close = None, None, None
        recent = [c for (c,) in self.recent.get()] if self.recent.exists() else []

        # 2. process rows in date order
        out = []
        for row in batch.itertuples(index=False):
            close, signal = float(row.close), None
            if peak_close is None or close > peak_close:
                if peak_close is not None:
                    signal = "NEW HIGH"
                peak_close, peak_date = close, row.date
            else:
                prev_level = int((1 - last_close / peak_close) * 10)   # 0 = within 10% of peak, 1 = 10-20% down, ...
                level = int((1 - close / peak_close) * 10)
                if level > prev_level:
                    signal = f"DRAWDOWN {level * 10}%"

            recent = (recent + [close])[-WINDOW:]
            sma = round(sum(recent) / WINDOW, 2) if len(recent) == WINDOW else None
            drawdown = round((close / peak_close - 1) * 100, 1)
            out.append((ticker, row.date, close, sma, peak_close, peak_date, drawdown, signal))
            last_close = close

        # 3. save state for the next micro-batch
        self.peak.update((peak_close, peak_date, last_close))
        self.recent.put([(c,) for c in recent])

        yield pd.DataFrame(out, columns=OUTPUT_SCHEMA.fieldNames())

    def close(self) -> None:
        pass

**Point out:** the processor never sees the whole history, only this batch's rows plus whatever it chose to remember. Five floats and a few scalars per ticker is all the state this job needs, no matter how long it runs.

---

### 3. The Streaming Query

`groupBy("ticker")` sends each ticker's rows to our processor. `trigger(availableNow=True)` processes every file currently in the landing folder (one per batch) and then stops, which makes it easy to run in a notebook.

In [ ]:
INPUT_SCHEMA = "date STRING, ticker STRING, close DOUBLE, adjusted_close DOUBLE, volume LONG"

def run_query():
    prices = (spark.readStream
        .schema(INPUT_SCHEMA)
        .option("header", True)
        .option("maxFilesPerTrigger", 1)      # one month per micro-batch
        .csv(LANDING))

    signals = (prices
        .groupBy("ticker")
        .transformWithStateInPandas(
            statefulProcessor=DrawdownTracker(),
            outputStructType=OUTPUT_SCHEMA,
            outputMode="Append",
            timeMode="None"))

    query = (signals.writeStream
        .format("parquet")
        .option("path", OUTPUT)
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .start())
    query.awaitTermination()

    for p in query.recentProgress:
        if p["numInputRows"] > 0:
            state_rows = p["stateOperators"][0]["numRowsTotal"] if p["stateOperators"] else None
            print(f"batch {p['batchId']:2d}: {p['numInputRows']:3d} input rows, state rows = {state_rows}")

#### Run 1: the first 6 months

In [ ]:
land_months(MONTHS[:6])
run_query()

In [ ]:
results = spark.read.parquet(OUTPUT)

print("signals so far:")
results.filter("signal IS NOT NULL").orderBy("ticker", "date").show(50)

**Point out:** each month was a separate micro-batch, yet the peak and drawdown levels carry smoothly across month boundaries. That continuity is the state at work: without it, every batch would start over with the first close of the month as the "peak".

#### Run 2: restart the query with the rest of the year

The query from run 1 has **stopped**. We land the remaining months and start a brand new query object with the **same checkpoint**. Spark reads the checkpoint to find which files it already processed and restores each ticker's state from RocksDB.

In [ ]:
land_months(MONTHS[6:])
run_query()

In [ ]:
results = spark.read.parquet(OUTPUT)
boundary = MONTHS[6]

# per ticker: the last trading day before the restart and the first one after
last_before = results.filter(F.col("date") < boundary).groupBy("ticker").agg(F.max("date").alias("date"))
first_after = results.filter(F.col("date") >= boundary).groupBy("ticker").agg(F.min("date").alias("date"))

print(f"either side of the restart ({boundary}):")
(results.join(last_before.union(first_after), ["ticker", "date"])
    .select("ticker", "date", "close", "sma_5", "peak_close", "peak_date", "drawdown_pct")
    .orderBy("ticker", "date")
    .show())

**Point out:** the first day after the restart still knows the peak from months earlier, and its `sma_5` is defined immediately because the last 5 closes came back from the `ListState`. Spark also skipped the 6 files it had already processed. This is the same checkpoint mechanism that gives streaming jobs fault tolerance: a crashed job restarts exactly where it left off.

#### All signals for the year

In [ ]:
results.filter("signal IS NOT NULL").orderBy("ticker", "date").show(100)

(results.groupBy("ticker")
    .agg(F.count("*").alias("days"),
         F.min("drawdown_pct").alias("max_drawdown_pct"),
         F.count("signal").alias("signals"))
    .show())

---

### 4. Beyond This Demo: Timers and TTL

Two features we didn't use, both declared through the same `handle`:

- **Timers**: with `timeMode="ProcessingTime"` or `"EventTime"`, call `handle.registerTimer(ms)` and Spark calls `handleExpiredTimer(key, timerValues, expiredTimerInfo)` when it fires, even if no new rows arrive for that key. Use it for things like *"alert if a ticker sends no quotes for 5 minutes."*
- **TTL**: `handle.getValueState("peak", PEAK_SCHEMA, ttlDurationMs=...)` makes Spark expire state that hasn't been updated, so state for keys that disappear (delisted tickers, finished sessions) doesn't grow forever.

Before Spark 4.0, this kind of logic used `applyInPandasWithState`, which allowed only one state object per key and no TTL. transformWithState replaces it.

---

### TRY IT

1. Delete the `checkpoint` folder (but not `signals`) and rerun run 2. What happens to the peaks, and why are there now duplicate rows in the output?
2. Add a `MapState` keyed by month that counts the up days and down days per ticker, and add the counts to the output.
3. Add a signal for when the close crosses above or below `sma_5`. What extra state do you need?
4. Use `adjusted_close` instead of `close`. Do MSFT's signals change? Why (hint: dividends)?
5. Read the state directly: `spark.read.format("statestore").option("stateVarName", "peak").load(CHECKPOINT).show(truncate=False)`

In [ ]:
spark.stop()